In [ ]:
# 必要なものを用意
## ライブラリ
!pip install scikit-learn
## データセット
from datasets import load_dataset
clinc = load_dataset("clinc_oos", "plus")
## 評価関数
from sklearn.metrics import accuracy_score
## パイプライン
from transformers import pipeline
bert_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=bert_ckpt)
## 意図
intents = clinc["test"].features["intent"]

In [ ]:
# ハイパーパラメータの追加
from transformers import TrainingArguments

class DistillationTrainingArguments(TrainingArguments):
    def __init__(self, *args, alpha=0.5, temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha
        self.temperature = temperature

In [ ]:
# カスタムトレーナークラス
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.teacher_model.eval()  # 教師モデルは評価モードにする

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs_student = model(**inputs)
        # 生徒からクロスエントロピー損失とロジット抽出
        loss_ce = outputs_student.loss
        logits_student = outputs_student.logits
        # 教師からロジット抽出
        with torch.no_grad():
            outputs_teacher = self.teacher_model(**inputs)
            logits_teacher = outputs_teacher.logits
        # 確率をソフト化し蒸留損失を計算
        loss_function = nn.KLDivLoss(reduction='batchmean') # バッチの次元で損失を平均化
        loss_kd = self.args.temperature ** 2 * loss_function(
            F.log_softmax(logits_student / self.args.temperature, dim=-1),
            F.softmax(logits_teacher / self.args.temperature, dim=-1))
        # 重み付き損失
        loss = self.args.alpha * loss_ce + (1. - self.args.alpha) * loss_kd
        return (loss, outputs_student) if return_outputs else loss

In [ ]:
# クエリの前処理
from transformers import AutoTokenizer

student_ckpt = "distilbert-base-uncased"
student_tokenizer = AutoTokenizer.from_pretrained(student_ckpt)

def tokenize_text(batch):
    return student_tokenizer(batch["text"], truncation=True)

clinc_enc = clinc.map(tokenize_text, batched=True, remove_columns=["text"])
clinc_enc = clinc_enc.rename_column("intent", "labels")

In [ ]:
# Hugging Face ログイン
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# 評価指標の定義
import numpy as np

def compute_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [ ]:
# α = 1 (教師なし)蒸留
batch_size = 48

finetuned_ckpt = "distilbert-base-uncased-finetuned-clinc"
student_training_args = DistillationTrainingArguments(
    output_dir=finetuned_ckpt, eval_strategy = "epoch",
    num_train_epochs=5, learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    alpha=1, weight_decay=0.01,
    push_to_hub=True)

In [ ]:
# 意図とラベルのマッピング
id2label = pipe.model.config.id2label
label2id = pipe.model.config.label2id

In [ ]:
# カスタムモデルの設定
from transformers import AutoConfig

num_labels = intents.num_classes
student_config = (AutoConfig
                  .from_pretrained(student_ckpt, num_labels=num_labels,
                                   id2label=id2label, label2id=label2id))

In [ ]:
# 生徒モデル初期化
from transformers import AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def student_init():
    return (AutoModelForSequenceClassification
            .from_pretrained(student_ckpt, config=student_config).to(device))

In [ ]:
# ファインチューニング実行
from transformers import DataCollatorWithPadding

teacher_model = (AutoModelForSequenceClassification
                 .from_pretrained(bert_ckpt, num_labels=num_labels)
                 .to(device))

data_collator = DataCollatorWithPadding(tokenizer=student_tokenizer)

distilbert_trainer = DistillationTrainer(model_init=student_init,
    teacher_model=teacher_model, args=student_training_args,
    train_dataset=clinc_enc["train"], eval_dataset=clinc_enc["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator, processing_class=student_tokenizer)

distilbert_trainer.train()

In [ ]:
# モデルを Hub にプッシュ
distilbert_trainer.push_to_hub("Training completed!")